In [ ]:
# ============================================================
# 02_feature_extraction.ipynb  —  ATLAS GO Prediction Pipeline
# Extracts: ProtBERT (P), ESM2 (E), Static (S), Dynamic (D)
# Creates:  9 feature combination files for model training
# Run after: 01_data_preparation.ipynb
# ============================================================

In [1]:
## ── 1. Mount + Install Dependencies ─────────────────────────
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
# Install required packages
!pip install fair-esm --quiet
!pip install transformers sentencepiece --quiet
!pip install tqdm joblib --quiet
print("Packages installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 10.1 MB/s eta 0:00:00
Packages installed!


In [3]:
## ── 2. Imports + Config ──────────────────────────────────────
import yaml
import os
import numpy as np
import pandas as pd
import torch
import joblib
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Load config
cfg_path = "/content/drive/MyDrive/atlas-go-dynamicsV2/configs/config.yaml"
with open(cfg_path, "r") as f:
    cfg = yaml.safe_load(f)

data_raw       = cfg['paths']['data_raw']
data_processed = cfg['paths']['data_processed']
SEED           = cfg['random_seed']
S_COLS         = cfg['data']['static_features']   # ['α%', 'β%', 'Coil%', 'Len.']
D_COLS         = cfg['data']['dynamic_features']  # ['Avg. RMSF', 'Avg. gyr.', 'Div. SE', 'Div. MM']
MAX_SEQ_LEN    = cfg['data']['esm2_max_seq_len']  # 1000

# CUDA check
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"Config loaded. S_cols: {S_COLS}")
print(f"D_cols: {D_COLS}")

Using device: cuda
Config loaded. S_cols: ['α%', 'β%', 'Coil%', 'Len.']
D_cols: ['Avg. RMSF', 'Avg. gyr.', 'Div. SE', 'Div. MM']


In [4]:
## ── 3. Load Full Dataset ─────────────────────────────────────
df = pd.read_csv(f"{data_processed}/full_dataset.csv")

# Verify required columns exist
required_cols = ['UniProt', 'Sequence'] + S_COLS + D_COLS
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in full_dataset.csv: {missing}")

print(f"Loaded {len(df)} proteins")
print(f"Columns: {df.columns.tolist()}")
print(f"Sample sequence (first 50 aa): {df['Sequence'].iloc[0][:50]}")
print(f"Sequence lengths: min={df['Sequence'].str.len().min()}, "
      f"max={df['Sequence'].str.len().max()}, "
      f"mean={df['Sequence'].str.len().mean():.0f}")

Loaded 1289 proteins
Columns: ['PDB', 'UniProt', 'α%', 'β%', 'Coil%', 'Len.', 'Avg. RMSF', 'Avg. gyr.', 'Div. SE', 'Div. MM', 'Sequence', 'MF', 'BP', 'CC', 'MFO_clean', 'BPO_clean', 'CCO_clean', 'all_GO']
Sample sequence (first 50 aa): MTLNEKKSINECDLKGKKVLIRVDFNVPVKNGKITNDYRIRSALPTLKKV
Sequence lengths: min=46, max=7176, mean=532


In [5]:
## ── 4. Extract ProtBERT Embeddings (P, 1024-dim) ─────────────
# NOTE: ProtBERT requires space-separated amino acids as input
from transformers import AutoTokenizer, AutoModel

print("Loading ProtBERT model...")
tokenizer_p = AutoTokenizer.from_pretrained(
    "Rostlab/prot_bert", do_lower_case=False, trust_remote_code=True
)
model_p = AutoModel.from_pretrained(
    "Rostlab/prot_bert", trust_remote_code=True
)
model_p = model_p.to(device)
model_p.eval()
print(f"ProtBERT loaded on {device}")

def get_protbert_embeddings(sequences, batch_size=32):
    """Extract mean-pooled ProtBERT embeddings. Sequences must be space-separated AA."""
    # ProtBERT requires space-separated amino acids
    spaced_seqs = [" ".join(list(seq[:1024])) for seq in sequences]
    embeddings = []
    for i in tqdm(range(0, len(spaced_seqs), batch_size), desc="ProtBERT", unit="batch"):
        batch = spaced_seqs[i : i + batch_size]
        inputs = tokenizer_p(
            batch,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=1024
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model_p(**inputs)
        # Mean pool over sequence length (exclude [CLS] and [SEP] tokens)
        # Shape: (batch, seq_len, 1024) -> (batch, 1024)
        attention_mask = inputs['attention_mask'].unsqueeze(-1).float()
        token_emb = outputs.last_hidden_state  # (batch, seq_len, 1024)
        # Mask out padding and special tokens (index 0 = [CLS], -1 = [SEP])
        sum_emb = (token_emb * attention_mask).sum(dim=1)
        count   = attention_mask.sum(dim=1).clamp(min=1)
        emb = (sum_emb / count).cpu().numpy()
        embeddings.append(emb)
        torch.cuda.empty_cache()
    return np.vstack(embeddings)

P_emb = get_protbert_embeddings(df['Sequence'].tolist())
np.save(f"{data_processed}/X_p.npy", P_emb)
print(f"\nProtBERT COMPLETE: {P_emb.shape}")
print(f"Saved: {data_processed}/X_p.npy")

# Free GPU memory before loading ESM2
del model_p
torch.cuda.empty_cache()

Loading ProtBERT model...


config.json:   0%|          | 0.00/361 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/81.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/487 [00:00<?, ?it/s]

BertModel LOAD REPORT from: Rostlab/prot_bert
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ProtBERT loaded on cuda



ProtBERT: 100%|██████████| 41/41 [05:57<00:00,  8.72s/batch]


ProtBERT COMPLETE: (1289, 1024)
Saved: /content/drive/MyDrive/atlas-go-dynamicsV2/data/processed/X_p.npy


In [6]:
## ── 5. Extract ESM2 Embeddings (E, 1280-dim) ─────────────────
import esm  # fair-esm

print("Loading ESM2 model (esm2_t33_650M_UR50D)...")
esm_model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
esm_model = esm_model.to(device)
esm_model.eval()
batch_converter = alphabet.get_batch_converter()
print(f"ESM2 loaded on {device}")

def get_esm2_embeddings(sequences, batch_size=8):
    """Extract mean-pooled ESM2 embeddings (layer 33)."""
    embeddings = []
    for i in tqdm(range(0, len(sequences), batch_size), desc="ESM2", unit="batch"):
        batch_seqs = sequences[i : i + batch_size]
        # Truncate to max_seq_len to avoid OOM
        data = [(f"protein_{i+j}", seq[:MAX_SEQ_LEN]) for j, seq in enumerate(batch_seqs)]
        batch_labels, batch_strs, batch_tokens = batch_converter(data)
        batch_tokens = batch_tokens.to(device)
        with torch.no_grad():
            results = esm_model(
                batch_tokens,
                repr_layers=[33],
                return_contacts=False
            )
        token_reps = results["representations"][33]  # (batch, seq_len+2, 1280)
        # Exclude [BOS] (index 0) and [EOS] (index -1), then mean pool
        for j in range(token_reps.shape[0]):
            seq_len = len(batch_strs[j])  # actual (truncated) sequence length
            emb = token_reps[j, 1 : seq_len + 1].mean(0).cpu().numpy()
            embeddings.append(emb)
        torch.cuda.empty_cache()
    return np.array(embeddings)

E_emb = get_esm2_embeddings(df['Sequence'].tolist())
np.save(f"{data_processed}/X_e.npy", E_emb)
print(f"\nESM2 COMPLETE: {E_emb.shape}")
print(f"Saved: {data_processed}/X_e.npy")

# Free GPU memory
del esm_model
torch.cuda.empty_cache()

Loading ESM2 model (esm2_t33_650M_UR50D)...
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t33_650M_UR50D.pt" to /root/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t33_650M_UR50D-contact-regression.pt" to /root/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D-contact-regression.pt
ESM2 loaded on cuda


ESM2: 100%|██████████| 162/162 [10:41<00:00,  3.96s/batch]



ESM2 COMPLETE: (1289, 1280)
Saved: /content/drive/MyDrive/atlas-go-dynamicsV2/data/processed/X_e.npy


In [7]:
## ── 6. Verify Embeddings ─────────────────────────────────────
print("Verifying saved embeddings...")
for fname, expected_dim in [('X_p.npy', 1024), ('X_e.npy', 1280)]:
    path = f"{data_processed}/{fname}"
    if os.path.exists(path):
        X = np.load(path)
        size_mb = X.nbytes / 1e6
        assert X.shape[0] == len(df), f"Row mismatch in {fname}: {X.shape[0]} vs {len(df)}"
        assert X.shape[1] == expected_dim, f"Dim mismatch in {fname}: {X.shape[1]} vs {expected_dim}"
        print(f"  {fname}: {X.shape} | {size_mb:.1f} MB | OK")
    else:
        print(f"  {fname}: MISSING!")
print("Embeddings verified!")

Verifying saved embeddings...
  X_p.npy: (1289, 1024) | 5.3 MB | OK
  X_e.npy: (1289, 1280) | 6.6 MB | OK
Embeddings verified!


In [8]:
## ── 7. Normalize S (Static) + D (Dynamic) Features ───────────
# CRITICAL: Fit scaler on TRAIN split ONLY to prevent data leakage
print("Normalizing S+D features (train-fit only)...")

# Load split indices
splits    = np.load(f"{data_processed}/splits.npz")
train_idx = splits['train_idx']

# Extract raw feature arrays
S_raw = df[S_COLS].values.astype(np.float32)  # (N, 4)
D_raw = df[D_COLS].values.astype(np.float32)  # (N, 4)

# Fit scalers on TRAIN only
scaler_S = StandardScaler()
scaler_D = StandardScaler()
scaler_S.fit(S_raw[train_idx])
scaler_D.fit(D_raw[train_idx])

# Transform ALL proteins using train-fitted scalers
S_norm = scaler_S.transform(S_raw).astype(np.float32)  # (N, 4)
D_norm = scaler_D.transform(D_raw).astype(np.float32)  # (N, 4)

# Save
np.save(f"{data_processed}/S_norm.npy", S_norm)
np.save(f"{data_processed}/D_norm.npy", D_norm)
joblib.dump(scaler_S, f"{data_processed}/scaler_S.pkl")
joblib.dump(scaler_D, f"{data_processed}/scaler_D.pkl")

print(f"  S_norm: {S_norm.shape} | mean={S_norm.mean():.3f}, std={S_norm.std():.3f}")
print(f"  D_norm: {D_norm.shape} | mean={D_norm.mean():.3f}, std={D_norm.std():.3f}")
print(f"  Scalers fitted on {len(train_idx)} train proteins (no leakage)")
print("S+D normalization complete!")

Normalizing S+D features (train-fit only)...
  S_norm: (1289, 4) | mean=0.001, std=0.998
  D_norm: (1289, 4) | mean=0.003, std=0.996
  Scalers fitted on 1031 train proteins (no leakage)
S+D normalization complete!


In [9]:
## ── 8. Build 9 Feature Combination Files ─────────────────────
# Load all base features
P = np.load(f"{data_processed}/X_p.npy")      # (N, 1024)
E = np.load(f"{data_processed}/X_e.npy")      # (N, 1280)
S = np.load(f"{data_processed}/S_norm.npy")   # (N, 4)
D = np.load(f"{data_processed}/D_norm.npy")   # (N, 4)

print(f"Base feature shapes:")
print(f"  P (ProtBERT):  {P.shape}")
print(f"  E (ESM2):      {E.shape}")
print(f"  S (Static):    {S.shape}")
print(f"  D (Dynamic):   {D.shape}")

# Define 9 combinations
combos = {
    'p':    P,
    'ps':   np.hstack([P, S]),
    'psd':  np.hstack([P, S, D]),
    'e':    E,
    'es':   np.hstack([E, S]),
    'esd':  np.hstack([E, S, D]),
    'pe':   np.hstack([P, E]),
    'pes':  np.hstack([P, E, S]),
    'pesd': np.hstack([P, E, S, D]),
}

print(f"\nSaving 9 feature combination files...")
for name, X in combos.items():
    X = X.astype(np.float32)
    np.save(f"{data_processed}/X_{name}.npy", X)
    print(f"  X_{name}: {X.shape} | {X.nbytes/1e6:.1f} MB")

print("\n9 FEATURE SETS SAVED!")

Base feature shapes:
  P (ProtBERT):  (1289, 1024)
  E (ESM2):      (1289, 1280)
  S (Static):    (1289, 4)
  D (Dynamic):   (1289, 4)

Saving 9 feature combination files...
  X_p: (1289, 1024) | 5.3 MB
  X_ps: (1289, 1028) | 5.3 MB
  X_psd: (1289, 1032) | 5.3 MB
  X_e: (1289, 1280) | 6.6 MB
  X_es: (1289, 1284) | 6.6 MB
  X_esd: (1289, 1288) | 6.6 MB
  X_pe: (1289, 2304) | 11.9 MB
  X_pes: (1289, 2308) | 11.9 MB
  X_pesd: (1289, 2312) | 11.9 MB

9 FEATURE SETS SAVED!


In [10]:
## ── 9. Final Verification ────────────────────────────────────
features = ['p', 'ps', 'psd', 'e', 'es', 'esd', 'pe', 'pes', 'pesd']
n_proteins = len(df)

print(f"{'='*55}")
print(" FEATURE EXTRACTION COMPLETE")
print(f"{'='*55}")
print(f"Proteins: {n_proteins}")
print()
all_ok = True
for name in features:
    path = f"{data_processed}/X_{name}.npy"
    if os.path.exists(path):
        X = np.load(path)
        ok = X.shape[0] == n_proteins and not np.isnan(X).any()
        status = "OK" if ok else "ERROR"
        all_ok = all_ok and ok
        print(f"  X_{name:<5}: {str(X.shape):<15} NaN={np.isnan(X).any()!s:<5} [{status}]")
    else:
        print(f"  X_{name}: MISSING!")
        all_ok = False

print()
if all_ok:
    print(" ALL FILES VERIFIED — run 03_training_evaluation.ipynb next")
else:
    print(" WARNING: Some files have issues, check above")

 FEATURE EXTRACTION COMPLETE
Proteins: 1289

  X_p    : (1289, 1024)    NaN=False [OK]
  X_ps   : (1289, 1028)    NaN=False [OK]
  X_psd  : (1289, 1032)    NaN=False [OK]
  X_e    : (1289, 1280)    NaN=False [OK]
  X_es   : (1289, 1284)    NaN=False [OK]
  X_esd  : (1289, 1288)    NaN=False [OK]
  X_pe   : (1289, 2304)    NaN=False [OK]
  X_pes  : (1289, 2308)    NaN=False [OK]
  X_pesd : (1289, 2312)    NaN=False [OK]

 ALL FILES VERIFIED — run 03_training_evaluation.ipynb next
